# Beyond BB84: E91, six-state QKD and entanglement swapping

Three protocol upgrades for the grid design, each checked against a number theory predicts in advance:

1. **E91 (entanglement + Bell test).** Alice and Bob share entangled pairs and compute the CHSH score *S*. Untouched pairs give *S* ≈ 2.83; any eavesdropper who measures the pairs pushes *S* to 2 or below, the most a classical (copied) signal can reach.
2. **Six-state QKD.** BB84 with a third basis. A full intercept-resend tap raises the error rate to 1/3 instead of BB84's 1/4, so the 11% alarm trips with more margin.
3. **Entanglement swapping.** A relay joins two entangled links into one end-to-end link *without ever holding the key*, the step that would replace trusted relays.

Each circuit packs several independent rounds side by side, so one job covers every setting. Builders and scorers are in `network_design/qkd2.py`.

> **Kernel:** qBraid-SDK. The GPU simulator run is a few seconds per job.

In [ ]:
import importlib.util, json, sys
if importlib.util.find_spec("qbraid") is None:
    raise ImportError("Wrong kernel: pick qBraid-SDK at the top right, then run this cell again.")
import numpy as np, matplotlib.pyplot as plt
from qiskit import QuantumCircuit
sys.path.insert(0, "network_design")
from qkd2 import *
import random

## 1. Build the circuits

- **E91:** 4 entangled pairs, one per CHSH setting (Alice at 0° or 90°, Bob at ±45°, all in the X–Z plane).
  - *tap*: Eve copies Bob's qubit in the Z basis, as in our BB84 attack model.
  - *blinding*: Eve measures each qubit in Bob's own basis. A detector-blinding attack lets her do this, but Bob's detector only clicks when her basis guess matches his (half the rounds).
- **Six-state vs BB84:** 12 sifted rounds each. Eve's basis differs from Alice's on 8 of 12 six-state rounds (expected error rate 8 × ½ / 12 = 1/3) and on 6 of 12 BB84 rounds (expected 1/4).
- **Swapping:** Alice–relay–Bob (and Alice–relay–relay–Bob). Each relay does a Bell measurement and the corrections are applied coherently. Then CHSH is measured across the whole chain. A small extra circuit records what the relay sees.

Only the qubits that are scored are measured, which keeps the result list short.

In [ ]:
import re
def only(qasm, keep):   # measure only the listed qubits
    return "\n".join(l for l in qasm.split("\n") if not l.startswith("measure") or int(re.search(r"q\[(\d+)\]", l).group(1)) in keep)
random.seed(2026)
B6 = ["Z","X","Y"]*4; bits6 = [random.randint(0,1) for _ in range(12)]
E6 = [B6[i] if i % 3 == 0 else {"Z":"X","X":"Y","Y":"Z"}[B6[i]] for i in range(12)]
B4 = ["Z","X"]*6; bits4 = [random.randint(0,1) for _ in range(12)]
E4 = [B4[i] if i % 2 == 0 else {"Z":"X","X":"Z"}[B4[i]] for i in range(12)]
s1, pairs1, relay1 = swap_chsh(1); s2, pairs2, relay2 = swap_chsh(2)
relay_view = "\n".join(['OPENQASM 2.0;','include "qelib1.inc";','qreg q[4];','creg c[4];','h q[0];','cx q[0],q[1];','h q[2];','cx q[2],q[3];',
                        'cx q[1],q[2];','h q[1];','cx q[2],q[3];','cz q[1],q[3];'] + [f'measure q[{i}] -> c[{i}];' for i in range(4)])
QASM = {"e91_quiet": e91("quiet"), "e91_tap": only(e91("tap"), range(8)), "e91_blind": e91("blind"),
        "six_quiet": prep_measure(B6, bits6), "six_tap": only(prep_measure(B6, bits6, E6), range(12)), "bb84_tap": only(prep_measure(B4, bits4, E4), range(12)),
        "swap1": only(s1, [q for p in pairs1 for q in p]), "swap2": only(s2, [q for p in pairs2 for q in p]), "swap_relay": relay_view}
circuits = {k: QuantumCircuit.from_qasm_str(v) for k, v in QASM.items()}
for k, c in circuits.items(): print(f"{k:11s} {c.num_qubits:2d} qubits, {c.count_ops().get('cx',0)+c.count_ops().get('cz',0):2d} two-qubit gates")

## 2. Run on qBraid

In [ ]:
from qbraid.runtime import QbraidProvider
import warnings; warnings.filterwarnings("ignore", message="Circuit contains measurement gates")
provider = QbraidProvider()
BACKENDS = [
    ("qbraid:qbraid:sim:gpu-sv", 35, {}),                                               # noise-free; what WRITEUP.md reports
    # ("ionq:ionq:sim:simulator", 29, {"runtime_options": {"noise_model": "forte-1"}}),  # free, IonQ Forte gate noise
    # ("quantinuum:quantinuum:sim:h2-emulator", 56, {}),                                 # H2 noise model
    # ("openquantum:ionq:qpu:forte-1", 36, {}),        # REAL HARDWARE: check with your team / a mentor first
]
SHOTS = 2000
counts, jobs = {}, {}
for dev_id, max_q, opts in BACKENDS:
    dev = provider.get_device(dev_id)
    for name, qc in circuits.items():
        if qc.num_qubits > max_q: print(f"skip {name} on {dev_id} ({qc.num_qubits} qubits)"); continue
        job = dev.run(qc, shots=SHOTS, **opts)
        counts[(dev_id, name)] = job.result().data.get_counts(); jobs[(dev_id, name)] = job.id
        print(f"{dev_id:40s} {name:11s} {job.id}")

## 3. Score

In [ ]:
rows = []
for (dev_id, name), c in counts.items():
    if name == "e91_blind":
        S_click, _ = chsh(c); S_all, _ = chsh(c, yield_=0.5)
        rows.append((dev_id, name, f"S = {S_click:.2f} on clicked rounds; S = {S_all:.2f} counting no-clicks (yield 50%); Eve matches Bob {np.mean([eve_match(c,k) for k in range(4)]):.0%}"))
    elif name.startswith("e91"): rows.append((dev_id, name, f"S = {chsh(c)[0]:.2f}"))
    elif name.startswith("six"): rows.append((dev_id, name, f"QBER = {qber(c, bits6):.3f}"))
    elif name == "bb84_tap": rows.append((dev_id, name, f"QBER = {qber(c, bits4):.3f}"))
    elif name in ("swap1", "swap2"): rows.append((dev_id, name, f"S = {chsh(c, pairs1 if name=='swap1' else pairs2)[0]:.2f} across {name[-1]} relay(s)"))
    elif name == "swap_relay":
        tot = sum(c.values()); agree = sum(v for k, v in c.items() if k.replace(' ','')[-1] == k.replace(' ','')[-4]) / tot
        rows.append((dev_id, name, f"Alice and Bob agree {agree:.1%}; relay learns {relay_info(c,[0],[[1,2]]):.4f} bits about the key bit"))
for r in rows: print(f"{r[0]:40s} {r[1]:11s} {r[2]}")

## 4. Chart

Bell scores (*S*) against the two limits: 2 (the most a measured-and-resent or classical signal can reach) and 2√2 ≈ 2.83 (perfect entanglement). Error rates for the two prepare-and-measure protocols against the 11% alarm.

In [ ]:
dev = BACKENDS[0][0]
S = {"E91 quiet": chsh(counts[(dev,"e91_quiet")])[0], "E91 tapped": chsh(counts[(dev,"e91_tap")])[0],
     "Blinding\n(clicked only)": chsh(counts[(dev,"e91_blind")])[0], "Blinding\n(no-clicks counted)": chsh(counts[(dev,"e91_blind")], yield_=0.5)[0],
     "Swap, 1 relay": chsh(counts[(dev,"swap1")], pairs1)[0], "Swap, 2 relays": chsh(counts[(dev,"swap2")], pairs2)[0]}
Q = {"Six-state\nquiet": qber(counts[(dev,"six_quiet")], bits6), "Six-state\ntapped": qber(counts[(dev,"six_tap")], bits6), "BB84\ntapped": qber(counts[(dev,"bb84_tap")], bits4)}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [2, 1]})
col = ["#2563eb", "#e5484d", "#f59f00", "#e5484d", "#2563eb", "#2563eb"]
ax[0].bar(list(S), list(S.values()), color=col); ax[0].axhline(2, ls="--", c="#111318", lw=1); ax[0].axhline(2*2**.5, ls=":", c="#2563eb", lw=1)
ax[0].text(5.45, 2.03, "classical limit 2", ha="right", fontsize=8); ax[0].text(5.45, 2.86, "2√2", ha="right", fontsize=8, color="#2563eb")
for i, v in enumerate(S.values()): ax[0].text(i, v + 0.04, f"{v:.2f}", ha="center", fontsize=9)
ax[0].set_ylim(0, 3.2); ax[0].set_ylabel("CHSH score S"); ax[0].set_title("Bell test (E91) and entanglement swapping"); ax[0].tick_params(axis="x", labelsize=8)
ax[1].bar(list(Q), list(Q.values()), color=["#2563eb", "#e0528f", "#8f8aa3"]); ax[1].axhline(0.11, ls="--", c="#e5484d", lw=1)
ax[1].text(2.4, 0.12, "11% alarm", ha="right", fontsize=8, color="#e5484d")
for i, v in enumerate(Q.values()): ax[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax[1].set_ylim(0, 0.42); ax[1].set_ylabel("error rate (QBER)"); ax[1].set_title("Six-state vs BB84 under a full tap"); ax[1].tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.savefig("figures/3j_qkd_upgrades.png", dpi=150); plt.show()

**Reading it:**

- **E91 tap:** a full tap drops *S* from about 2.8 to about 1.4, far below 2. The pairs are no longer entangled, and Alice and Bob can tell without comparing any key bits.
- **Blinding:** on the rounds where Bob's detector clicks, *S* stays near 2.8, so blinding fools a Bell test that only counts clicks. Once the no-click rounds are counted (Bob records a fixed value), *S* falls to about 1.4. Like BB84's yield check, it is the missing clicks that give blinding away, but here they also break the security proof's main number.
- **Six-state:** the same tap gives about 0.33 instead of BB84's 0.25, three times the 11% alarm instead of about twice.
- **Swapping:** *S* stays near 2.8 across one or two relays, and the relay's own measurements carry essentially zero information about the key. Unlike a trusted relay, a hacked swapping relay has no key to leak.